# Set Up the SDK

Once per workspace, before any project setup: choose the **catalog** and **schema** the Genie Ontology Modeled Knowledge SDK uses (a **new** schema the SDK bundle creates is recommended, default `ontology_sdk`), the group that reads the SDK assets and a SQL warehouse. The notebook checks the catalog and your permissions, writes the SDK bundle's gitignored `.databricks/bundle/<target>/variable-overrides.json` and prints the exact deploy and skills publishing steps.

* `schema_mode` = `new` (recommended): targets `dev` (schema prefixed `dev_<user>_`) or `prod`. `existing`: the schema must exist; target `prod_existing_schema` adds only the `sdk_wheels` volume.
* Nothing is deployed, granted or created here. Edited overrides are kept unless `force` is `true`.
* Required inputs are empty by default; the notebook does nothing until they are set.

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
# Required inputs have empty defaults; sdk_schema recommends a NEW schema the SDK bundle creates.
dbutils.widgets.text("sdk_catalog", "", "sdk_catalog (existing catalog, required)")
dbutils.widgets.text("sdk_schema", "ontology_sdk", "sdk_schema (new schema recommended)")
dbutils.widgets.dropdown("schema_mode", "new", ["new", "existing"], "schema_mode")
dbutils.widgets.dropdown("target", "dev", ["dev", "prod"], "target (new mode; existing uses prod_existing_schema)")
dbutils.widgets.text("consumer_group", "", "consumer_group (required for prod targets)")
dbutils.widgets.text("warehouse_id", "", "warehouse_id (required)")
dbutils.widgets.dropdown("force", "false", ["false", "true"], "force overwrite of edited overrides")
dbutils.widgets.text("sdk_root", "", "sdk_root (default: this notebook's SDK checkout)")

In [0]:
import importlib.util
import json
import sys
from pathlib import Path


def _w(name: str) -> str:
    return dbutils.widgets.get(name).strip()


if _w("sdk_root"):
    sdk_root = Path(_w("sdk_root"))
else:
    nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    sdk_root = Path("/Workspace" + nb_path).parent.parent

# SDK setup runs before the wheel is published: use the installed package if present,
# else load setup_project.py (standard library only) from the SDK checkout as a module.
try:
    from genie_ontology_modeled_knowledge_sdk import setup_project as sp
except ImportError:
    _file = sdk_root / "src" / "genie_ontology_modeled_knowledge_sdk" / "setup_project.py"
    _spec = importlib.util.spec_from_file_location("gos_setup_project", _file)
    sp = importlib.util.module_from_spec(_spec)
    sys.modules[_spec.name] = sp  # dataclasses resolve their module here
    _spec.loader.exec_module(sp)
    print(f"Loaded setup_project from the SDK checkout: {_file}")
print(f"SDK root: {sdk_root}")

In [0]:
from databricks.sdk import WorkspaceClient

missing = [n for n in ("sdk_catalog", "sdk_schema", "warehouse_id") if not _w(n)]
if missing:
    print("Set the required widgets, then re-run: " + ", ".join(missing))
    dbutils.notebook.exit(json.dumps({"ok": False, "missing": missing}))

inputs = sp.SdkSetupInputs(
    catalog=_w("sdk_catalog"),
    schema=_w("sdk_schema"),
    schema_mode=_w("schema_mode"),
    consumer_group=_w("consumer_group"),
    warehouse_id=_w("warehouse_id"),
    target=_w("target") if _w("schema_mode") == "new" else "",
)
result = sp.setup_sdk(WorkspaceClient(), sdk_root, inputs, force=_w("force") == "true")

for c in result.checks:
    print(f"[{c.status:>6}] {c.name}: {c.detail}")
if result.overrides is not None:
    print(f"\noverrides: {result.overrides.action} -> {result.overrides.path}")

In [0]:
if result.ok:
    print("SDK setup is ready. Run these steps in a terminal (Databricks CLI, this workspace's profile):\n")
    for i, cmd in enumerate(result.commands, 1):
        print(f"{i}. {cmd}")
    print("\nThen set up each project with notebooks/init_ontology_project.")
else:
    print("Fix the checks marked 'action' above, then re-run this notebook. Nothing was deployed.")
dbutils.notebook.exit(json.dumps(result.to_dict()))